# Prepare shared stable sparse VAR data

This notebook creates or verifies the frozen `sparseVAR` bundles used by
all seven method notebooks in this directory. Data generation and acquisition
are kept separate from model fitting so every method reads byte-identical
observations and the same declared ground truth.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tempfile
import types
import urllib.request
import zipfile
import pickle
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np
import pandas as pd
import scipy.special

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.upstream import verify_upstream_revision

In [ ]:
DATASET_KEY = "sparseVAR"
DATASET_ROOT = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison" / DATASET_KEY
)

# Confirmatory grid. Every method notebook reads the resulting files.
VARIABLES = (5, 10, 20)
SAMPLES = (250, 500, 1000)
ORDERS = (1, 2, 5)
NOISES = ("gaussian", "laplace")
REPLICATES = 50
SEED_START = 10_000

command = [
    sys.executable,
    str(PROJECT_ROOT / "experiments" / "benchmarks" / "benchmark_causal_methods.py"),
    "--datasets-only",
    "--variables", ",".join(map(str, VARIABLES)),
    "--samples", ",".join(map(str, SAMPLES)),
    "--orders", ",".join(map(str, ORDERS)),
    "--noises", ",".join(NOISES),
    "--replicates", str(REPLICATES),
    "--seed-start", str(SEED_START),
    "--output-dir", str(DATASET_ROOT),
]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print(f"Prepared {len(list((DATASET_ROOT / 'datasets').glob('*.npz')))} bundles.")